In [ ]:
print("Startup Job Finder – discovery layer ready")

Setup

In [1]:
from tavily import TavilyClient
from dotenv import load_dotenv
import os

load_dotenv()
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Company discovery query

In [ ]:
query = (
  '"AI startup" '
  '("about us" OR "company" OR "who we are") '
  '-news -blog -article -wikipedia -linkedin -medium'
)

response = client.search(
    query=query,
    search_depth="advanced",
    max_results=20
)

for r in response["results"]:
    print("TITLE:", r["title"])
    print("URL:", r["url"])
    print("----")


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CAREER_KEYWORDS = [
    "career",
    "job",
    "join",
    "work with",
    "hiring",
]

BAD_HINTS = ["contact", "about", "privacy", "terms", "press", "blog"]

def looks_like_careers(text: str, href: str) -> bool:
    t = text.lower()
    h = href.lower()

    # reject anchors like "#Contact"
    if h.startswith("#"):
        return False

    # reject obvious non-careers pages
    if any(b in t or b in h for b in BAD_HINTS):
        return False

    # accept careers-ish
    return any(k in t or k in h for k in CAREER_KEYWORDS)


def find_careers_url(homepage: str):
    try:
        resp = requests.get(homepage, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.find_all("a", href=True):
            text = (a.get_text() or "").lower()
            href = a["href"].lower()

            if looks_like_careers(text, href):
                url = urljoin(homepage, href)
                return url.rstrip("/")

    except Exception as e:
        print("ERROR:", homepage, e)

    return None


In [ ]:
companies = [
    "https://alphaai.biz",
    "https://ai-nation.de",
    "https://apera.ai",
]

for c in companies:
    print(c, "→", find_careers_url(c))


In [6]:
import requests
from bs4 import BeautifulSoup

def extract_careers_text(careers_url: str) -> str:
    resp = requests.get(careers_url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    # remove junk
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")

    # basic cleanup
    lines = [line.strip() for line in text.splitlines()]
    lines = [l for l in lines if len(l) > 30]

    return "\n".join(lines[:200])  # cap: first ~200 lines


In [ ]:
careers_pages = [
    ("Alpha AI", "https://alphaai.biz/careers"),
    ("Apera AI", "https://apera.ai/careers"),
]

for name, url in careers_pages:
    print("====", name, "====")
    text = extract_careers_text(url)
    print(text[:1000])
    print("\n")


In [4]:
JOB_HINTS = [
    "engineer", "developer", "scientist", "researcher", "intern",
    "backend", "full stack", "full-stack", "machine learning", "ml", "ai"
]

def extract_jobish_lines(text: str, window: int = 4) -> str:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    keep = set()

    for i, line in enumerate(lines):
        low = line.lower()
        if any(h in low for h in JOB_HINTS):
            for j in range(max(0, i - window), min(len(lines), i + window + 1)):
                keep.add(j)

    out = [lines[i] for i in sorted(keep)]
    return "\n".join(out)


In [ ]:
for name, url in careers_pages:
    print("====", name, "JOBISH ====")
    text = extract_careers_text(url)
    jobish = extract_jobish_lines(text, window=4)
    print(jobish[:1500])
    print("\n")


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def analyze_careers(company: str, text: str) -> str:
    prompt = f"""
You are analyzing startup careers pages for a job discovery tool.

Company: {company}

Below is extracted text from the careers page.
Your task:
1. Decide if this company has roles relevant to:
   - AI engineer
   - Machine learning
   - backend / full stack (Python-heavy)
2. Ignore generic culture / benefits text.
3. Extract concrete roles if present.
4. Return structured JSON only.

Return JSON schema:
{{
  "relevant": boolean,
  "roles": [
    {{
      "title": string,
      "seniority": "intern" | "junior" | "mid" | "senior" | "unknown",
      "tech_stack": [string],
      "remote_friendly": boolean | "unknown",
      "summary": string
    }}
  ],
  "overall_summary": string
}}

TEXT:
{text}
"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content


In [9]:
alpha_text = extract_jobish_lines(
    extract_careers_text("https://alphaai.biz/careers")
)

print(analyze_careers("Alpha AI", alpha_text))


```json
{
  "relevant": true,
  "roles": [
    {
      "title": "Full Stack Web Developer - Intern",
      "seniority": "intern",
      "tech_stack": ["MEAN", "MERN", "Flask", "Django", "PostgreSQL", "Python", "Docker"],
      "remote_friendly": "unknown",
      "summary": "Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM."
    },
    {
      "title": "React Native / Flutter App Developer - Intern",
      "seniority": "intern",
      "tech_stack": ["Flutter", "React Native", "Python", "PostgreSQL", "SQLite", "TensorFlow Lite", "Docker"],
      "remote_friendly": "unknown",
      "summary": "Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite."
    }
  ],
  "overall_summary": "Alpha AI is seeking interns for roles in full stack web development and mobile app development, both of which involve relevant technologies and skills in AI and machine learning."
}
```


In [10]:
apera_text = extract_jobish_lines(
    extract_careers_text("https://apera.ai/careers")
)

print(analyze_careers("Apera AI", apera_text))


```json
{
  "relevant": true,
  "roles": [
    {
      "title": "Principal Machine Learning Applied Scientist",
      "seniority": "unknown",
      "tech_stack": ["AI", "ML", "deep learning"],
      "remote_friendly": "unknown",
      "summary": "Create new inventions in artificial intelligence and deep learning that will be used by the world's leading manufacturers."
    },
    {
      "title": "Principal Software Development Engineer",
      "seniority": "unknown",
      "tech_stack": ["software engineering", "cloud computing"],
      "remote_friendly": "unknown",
      "summary": "Work in a technically rich environment focusing on robotics, 3D math, simulation, computer vision, and optimization."
    }
  ],
  "overall_summary": "Apera AI offers roles in machine learning and software development, focusing on AI and robotics, with an emphasis on innovative problem-solving and collaboration."
}
```
